# Kahneman Framing × TRIBE v2 — Colab Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/akifnu/DSprojects/blob/main/tribev2/notebooks/Kahneman_Framing_RCT.ipynb)

**Ready to go:** no Hugging Face token, no API keys, no repo clone.

1. **Runtime → Change runtime type → T4 GPU** (free tier works)
2. **Runtime → Run all**

Tests whether loss-framed vs gain-framed **text** (read aloud) produces different cortical predictions from [facebook/tribev2](https://huggingface.co/facebook/tribev2).

In [ ]:
!pip install -q 'tribev2 @ git+https://github.com/facebookresearch/TRIBEv2.git' gtts scipy

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# 12 classic Kahneman-style gain/loss pairs (objectively equivalent wording)
FRAMING_PAIRS = [
  {"id": "asian_disease", "domain": "health",
   "gain": "Program A will save 200 people for certain. Program B has a one-third probability that all 600 people will be saved.",
   "loss": "Program C will result in 400 people dying for certain. Program D has a one-third probability that nobody will die."},
  {"id": "surgery", "domain": "health",
   "gain": "The operation has a 90 percent success rate. Nine out of ten patients recover fully.",
   "loss": "The operation has a 10 percent failure rate. One out of ten patients do not survive."},
  {"id": "money_wallet", "domain": "financial",
   "gain": "You can keep 20 dollars for sure, or gamble fifty-fifty to keep 30 dollars or only 10 dollars.",
   "loss": "You will lose 10 dollars for sure and keep 20, or gamble fifty-fifty to lose nothing and keep 30, or lose 20 and keep 10."},
  {"id": "credit_card", "domain": "financial",
   "gain": "Paying cash gives you a 1 dollar discount compared to the credit card price.",
   "loss": "Paying by credit card adds a 1 dollar surcharge compared to the cash price."},
  {"id": "employment", "domain": "economic",
   "gain": "The new policy will help 80 percent of workers keep their jobs next year.",
   "loss": "The new policy means 20 percent of workers will lose their jobs next year."},
  {"id": "beef", "domain": "consumer",
   "gain": "The label says the ground beef is 75 percent lean.",
   "loss": "The label says the ground beef is 25 percent fat."},
  {"id": "exam", "domain": "education",
   "gain": "You answered 60 percent of the exam questions correctly.",
   "loss": "You answered 40 percent of the exam questions incorrectly."},
  {"id": "vaccine", "domain": "health",
   "gain": "The vaccine caused no serious side effects in 95 percent of recipients.",
   "loss": "The vaccine caused mild side effects in 5 percent of recipients."},
  {"id": "treatment_85", "domain": "health",
   "gain": "This treatment works for 85 percent of patients.",
   "loss": "This treatment fails for 15 percent of patients."},
  {"id": "investment", "domain": "financial",
   "gain": "The fund gained value on 70 percent of trading days last year.",
   "loss": "The fund lost value on 30 percent of trading days last year."},
  {"id": "pollution", "domain": "environment",
   "gain": "The cleanup plan removes 40 percent of river pollution within five years.",
   "loss": "The cleanup plan leaves 60 percent of river pollution in place within five years."},
  {"id": "course_pass", "domain": "education",
   "gain": "72 percent of students passed the certification course on the first attempt.",
   "loss": "28 percent of students failed the certification course on the first attempt."},
]
print(f"{len(FRAMING_PAIRS)} scenario pairs loaded")

In [ ]:
from pathlib import Path
from gtts import gTTS

AUDIO_DIR = Path('/content/framing_audio')
AUDIO_DIR.mkdir(exist_ok=True)

for pair in FRAMING_PAIRS:
    for frame in ('gain', 'loss'):
        path = AUDIO_DIR / f"{pair['id']}_{frame}.mp3"
        if not path.exists():
            gTTS(pair[frame], lang='en').save(str(path))
print('Audio stimuli ready:', len(list(AUDIO_DIR.glob('*.mp3'))), 'files')

In [ ]:
import numpy as np
from tribev2 import TribeModel

CACHE = '/content/tribe_cache'
print('Loading TRIBE v2 (first run downloads ~1 GB)...')
model = TribeModel.from_pretrained('facebook/tribev2', cache_folder=CACHE, device='cuda')
print('Model ready')

In [ ]:
def predict_audio(audio_path):
    events = model.get_events_dataframe(audio_path=str(audio_path))
    preds, _ = model.predict(events=events, verbose=False)
    return np.asarray(preds)

def summarize(preds, t=5):
    t = min(t, preds.shape[0] - 1)
    return {
        'timesteps': preds.shape[0],
        'mean_abs': float(np.mean(np.abs(preds))),
        'peak_abs': float(np.mean(np.abs(preds[t]))),
    }

results = []
for pair in FRAMING_PAIRS:
    row = {'id': pair['id'], 'domain': pair['domain']}
    for frame in ('gain', 'loss'):
        path = AUDIO_DIR / f"{pair['id']}_{frame}.mp3"
        preds = predict_audio(path)
        stats = summarize(preds)
        row[f'{frame}_mean_abs'] = stats['mean_abs']
        row[f'{frame}_peak_abs'] = stats['peak_abs']
    row['loss_minus_gain'] = row['loss_mean_abs'] - row['gain_mean_abs']
    results.append(row)
    print(f"{pair['id']:20s}  loss-gain = {row['loss_minus_gain']:+.4f}")

print('Done:', len(results), 'pairs')

In [ ]:
import pandas as pd
from scipy import stats

df = pd.DataFrame(results)
display(df[['id', 'domain', 'gain_mean_abs', 'loss_mean_abs', 'loss_minus_gain']])

gain_vals = df['gain_mean_abs'].values
loss_vals = df['loss_mean_abs'].values
t_stat, p_val = stats.ttest_rel(loss_vals, gain_vals)
diff = loss_vals - gain_vals
cohens_dz = diff.mean() / diff.std(ddof=1)
n_aligned = int((diff > 0).sum())

print('\n--- Kahneman framing test ---')
print(f'Pairs: {len(df)}')
print(f'Loss > gain (mean |activation|): {n_aligned}/{len(df)} scenarios')
print(f'Mean difference (loss - gain): {diff.mean():.4f}')
print(f'Paired t-test p-value: {p_val:.4f}')
print(f"Cohen's dz: {cohens_dz:.3f}")
if diff.mean() > 0 and p_val < 0.05:
    print('\n✓ Direction matches Kahneman loss-salience (loss framing → stronger cortical magnitude)')
else:
    print('\n→ No significant Kahneman-aligned effect in this batch (try more pairs or full GPU run)')